# gold_agg

Cruza `post_hist` y `users_hist` de Silver y carga la tabla Delta `cant_post_x_user_hist`, con una fila por usuario y mes.
Solo entran los posts de usuarios creados en los mismos meses cargados, porque Users se filtra por fecha de creación desde Bronze.

## Paso 1 – Librerías

In [0]:
from datetime import datetime

import pytz
from delta.tables import DeltaTable
from pyspark.sql import functions as F

## Paso 2 – Parámetros

Se pueden cambiar desde el notebook o desde el Job. `meses` indica qué meses de posts se agregan.

In [0]:
dbutils.widgets.text("anio", "2021")
dbutils.widgets.text("meses", "1,2")
dbutils.widgets.text("storage_account", "stsbmprod001")
dbutils.widgets.text("catalogo_silver", "silversbm")
dbutils.widgets.text("catalogo_gold", "goldsbm")
dbutils.widgets.text("esquema", "stackoverflow")

anio = int(dbutils.widgets.get("anio"))
meses = [int(m) for m in dbutils.widgets.get("meses").split(",")]
storage_account = dbutils.widgets.get("storage_account")
catalogo_silver = dbutils.widgets.get("catalogo_silver")
catalogo_gold = dbutils.widgets.get("catalogo_gold")
esquema = dbutils.widgets.get("esquema")

if meses != list(range(meses[0], meses[-1] + 1)):
    raise ValueError(f"Los meses deben ser seguidos: {meses}")

tabla_post = f"{catalogo_silver}.{esquema}.post_hist"
tabla_users = f"{catalogo_silver}.{esquema}.users_hist"
tabla_gold = f"{catalogo_gold}.{esquema}.cant_post_x_user_hist"
ruta_gold = f"abfss://gold@{storage_account}.dfs.core.windows.net/stackoverflow/cant_post_x_user_hist"
lista_meses = ", ".join(str(m) for m in meses)
filtro_gold = f"post_year = {anio} AND post_month IN ({lista_meses})"

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo_gold}.{esquema}")
print(f"Año {anio}, meses {meses}")
print(tabla_gold)

Año 2021, meses [1, 2]
goldsbm.stackoverflow.cant_post_x_user_hist


## Paso 3 – Agregar

Se cruzan los posts con su dueño y se agrupan por usuario y mes. Se agrupa por mes y no por año, como en el Proyecto 2, para que con un solo año se vea el efecto de cargar el segundo mes.
`avg_score` se calcula con decimales para que dé igual en Spark y en DuckDB, porque con `double` los empates se redondean distinto.

In [0]:
gold_origen = spark.sql(f"""
    SELECT
        u.id                                      AS user_id,
        p.post_year,
        p.post_month,
        u.display_name,
        u.location,
        u.reputation,
        u.creation_year                           AS user_creation_year,
        u.creation_month                          AS user_creation_month,
        COUNT_IF(p.post_type_id = 1)              AS questions,
        COUNT_IF(p.post_type_id = 2)              AS answers,
        COUNT(*)                                  AS total_posts,
        SUM(p.score)                              AS total_score,
        CAST(ROUND(CAST(SUM(p.score) AS DECIMAL(38, 6)) / COUNT(p.score), 2) AS DECIMAL(18, 2)) AS avg_score,
        MAX(p.score)                              AS max_score,
        SUM(p.view_count)                         AS total_views,
        MIN(p.creation_date)                      AS first_post_date,
        MAX(p.creation_date)                      AS last_post_date,
        current_timestamp()                       AS fecha_cargue
    FROM {tabla_post} p
    JOIN {tabla_users} u ON p.owner_user_id = u.id
    WHERE p.post_year = {anio} AND p.post_month IN ({lista_meses})
    GROUP BY
        u.id, p.post_year, p.post_month, u.display_name, u.location, u.reputation,
        u.creation_year, u.creation_month
""")

filas_origen = gold_origen.count()
print(f"{filas_origen:,} filas por usuario y mes")
if filas_origen == 0:
    raise ValueError(f"No hay datos en Silver para {anio}, meses {meses}")

50,448 filas por usuario y mes


## Paso 4 – Validar y hacer merge

La tabla se crea vacía y externa la primera vez. Antes del merge se cuentan los registros nuevos y por actualizar, y después se comparan con las métricas que registró Delta.

In [0]:
llaves = ["user_id", "post_year", "post_month"]

if not spark.catalog.tableExists(tabla_gold):
    (gold_origen.limit(0).write.format("delta")
        .partitionBy("post_year", "post_month")
        .option("path", ruta_gold)
        .saveAsTable(tabla_gold))

filas_antes = spark.table(tabla_gold).count()
llaves_origen = gold_origen.select(*llaves)
llaves_destino = spark.table(tabla_gold).select(*llaves)
por_actualizar = llaves_origen.join(llaves_destino, on=llaves, how="inner").count()
nuevos = llaves_origen.join(llaves_destino, on=llaves, how="left_anti").count()
print(f"{tabla_gold}: {nuevos:,} nuevos, {por_actualizar:,} por actualizar")

condicion = " AND ".join(f"tgt.{c} = src.{c}" for c in llaves)
destino = DeltaTable.forName(spark, tabla_gold)
(destino.alias("tgt")
    .merge(gold_origen.alias("src"), condicion)
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

metricas = destino.history(10).filter("operation = 'MERGE'").orderBy(F.col("version").desc()).select("operationMetrics").first()[0]
insertados = int(metricas.get("numTargetRowsInserted", 0))
actualizados = int(metricas.get("numTargetRowsUpdated", 0))
filas_despues = spark.table(tabla_gold).count()
print(f"{tabla_gold}: merge con {insertados:,} insertados y {actualizados:,} actualizados, {filas_despues:,} filas en total")

if insertados != nuevos or actualizados != por_actualizar:
    raise ValueError("El merge no coincide con la validación")
if filas_despues != filas_antes + nuevos:
    raise ValueError(f"Había {filas_antes} filas y quedaron {filas_despues}, con {nuevos} nuevos")

goldsbm.stackoverflow.cant_post_x_user_hist: 0 nuevos, 50,448 por actualizar
goldsbm.stackoverflow.cant_post_x_user_hist: merge con 0 insertados y 50,448 actualizados, 50,448 filas en total


## Paso 5 – Revisar el resultado

Los posts sumados en Gold deben ser los mismos que da el cruce en Silver. Después se muestran las métricas por mes, los usuarios con más posts y el historial de Delta.

In [0]:
posts_silver = spark.sql(f"""
    SELECT COUNT(*)
    FROM {tabla_post} p
    JOIN {tabla_users} u ON p.owner_user_id = u.id
    WHERE p.post_year = {anio} AND p.post_month IN ({lista_meses})
""").first()[0]
posts_gold = spark.table(tabla_gold).filter(filtro_gold).agg(F.sum("total_posts")).first()[0]
print(f"Posts: {posts_silver:,} en Silver y {posts_gold:,} en Gold")
if posts_silver != posts_gold:
    raise ValueError("Los posts de Gold no coinciden con Silver")

repetidas = spark.table(tabla_gold).groupBy(*llaves).count().filter("count > 1").count()
print(f"{tabla_gold}: {repetidas} llaves repetidas")
if repetidas > 0:
    raise ValueError("Gold tiene llaves repetidas")

Posts: 75,849 en Silver y 75,849 en Gold
goldsbm.stackoverflow.cant_post_x_user_hist: 0 llaves repetidas


In [0]:
display(spark.sql(f"""
    SELECT
        post_year AS anio,
        post_month AS mes,
        COUNT(*) AS usuarios,
        SUM(questions) AS preguntas,
        SUM(answers) AS respuestas,
        SUM(total_posts) AS posts,
        SUM(total_score) AS score,
        SUM(total_views) AS vistas,
        MAX(fecha_cargue) AS fecha_cargue
    FROM {tabla_gold}
    GROUP BY 1, 2
    ORDER BY 1, 2
"""))

display(spark.table(tabla_gold).orderBy(F.col("total_posts").desc(), "user_id").limit(10))

anio,mes,usuarios,preguntas,respuestas,posts,score,vistas,fecha_cargue
2021,1,23736,23688,11094,34785,21763,18189871,2026-11-05T14:21:48.813Z
2021,2,26712,27784,13280,41064,25137,22670500,2026-11-05T14:21:48.813Z


user_id,post_year,post_month,display_name,location,reputation,user_creation_year,user_creation_month,questions,answers,total_posts,total_score,avg_score,max_score,total_views,first_post_date,last_post_date,fecha_cargue
15019536,2021,2,Meysam Asadi,null,6530,2021,1,0,135,135,60,0.44,3,0,2021-02-01T04:09:43.380Z,2021-02-28T18:24:04.203Z,2026-11-05T14:21:48.813Z
15102459,2021,2,Thiyanesh,null,2360,2021,1,0,110,110,136,1.24,6,0,2021-02-01T01:36:02.460Z,2021-02-28T07:20:54.700Z,2026-11-05T14:21:48.813Z
14991864,2021,1,Abdul Aziz Barkat,null,20649,2021,1,0,104,104,79,0.76,4,0,2021-01-12T15:33:46.187Z,2021-01-31T14:07:32.227Z,2026-11-05T14:21:48.813Z
14991864,2021,2,Abdul Aziz Barkat,null,20649,2021,1,0,103,103,115,1.12,6,0,2021-02-01T08:40:04.313Z,2021-02-28T17:47:32.160Z,2026-11-05T14:21:48.813Z
15019536,2021,1,Meysam Asadi,null,6530,2021,1,1,82,83,31,0.37,4,117,2021-01-16T16:27:35.053Z,2021-01-31T12:35:16.420Z,2026-11-05T14:21:48.813Z
14954503,2021,2,AdriSolid,null,2725,2021,1,1,73,74,62,0.84,5,3033,2021-02-01T17:38:47.963Z,2021-02-28T20:27:22.593Z,2026-11-05T14:21:48.813Z
14956277,2021,1,D M,null,6330,2021,1,0,64,64,85,1.33,8,0,2021-01-07T20:05:31.250Z,2021-01-29T22:44:15.740Z,2026-11-05T14:21:48.813Z
15051310,2021,1,Timothy Alexis Vass,"Cubelles, Barcelona, España",2619,2021,1,1,63,64,60,0.94,13,545,2021-01-22T09:18:36.323Z,2021-01-31T15:58:48.437Z,2026-11-05T14:21:48.813Z
14989571,2021,1,Shunya,"Barcelona, Spain",2354,2021,1,1,62,63,60,0.95,3,1002,2021-01-12T10:10:23.110Z,2021-01-23T19:08:18.210Z,2026-11-05T14:21:48.813Z
14941819,2021,1,Matt Miguel,"Los Angeles, CA, USA",1355,2021,1,0,62,62,68,1.10,9,0,2021-01-05T03:53:49.537Z,2021-01-31T19:15:46.863Z,2026-11-05T14:21:48.813Z


In [0]:
display(spark.sql(f"DESCRIBE HISTORY {tabla_gold}").select("version", "timestamp", "operation", "operationMetrics"))

zona = pytz.timezone("America/Bogota")
print(f"Gold terminado: {datetime.now(zona).strftime('%Y-%m-%d %H:%M:%S')} (hora Colombia)")

version,timestamp,operation,operationMetrics
4,2026-11-05T14:21:52.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2145987, numTargetBytesRemoved -> 2147407, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 50448, executionTimeMs -> 2984, materializeSourceTimeMs -> 805, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 792, numTargetRowsUpdated -> 50448, numOutputRows -> 50448, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 50448, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1360)"
3,2026-11-05T14:21:17.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1139754, numTargetBytesRemoved -> 1139754, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 26712, executionTimeMs -> 2390, materializeSourceTimeMs -> 703, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 640, numTargetRowsUpdated -> 26712, numOutputRows -> 26712, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 26712, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1021)"
2,2026-11-05T14:20:42.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1139754, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2415, materializeSourceTimeMs -> 751, numTargetRowsInserted -> 26712, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 859, numTargetRowsUpdated -> 0, numOutputRows -> 26712, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 26712, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 776)"
1,2026-11-05T14:20:06.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1007653, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2364, materializeSourceTimeMs -> 866, numTargetRowsInserted -> 23736, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 620, numTargetRowsUpdated -> 0, numOutputRows -> 23736, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 23736, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 853)"
0,2026-11-05T14:20:01.000Z,CREATE TABLE AS SELECT,"Map(numFiles -> 0, numOutputRows -> 0, numOutputBytes -> 0)"


Gold terminado: 2026-11-05 09:21:58 (hora Colombia)
